<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# 🎓 Platform Validation: Your AI Infrastructure Health Check

Welcome to your first step in building an **AI Research Assistant**! Before we dive into coding RAG pipelines and multi-agent systems, we need to ensure all platform services are operational.

## 🎯 What You'll Learn

In this notebook, you'll:
- ✅ **Validate 7 critical AI infrastructure services**
- 🧠 **Understand how each service contributes to AI applications**
- 🔗 **See how these services work together in a real AI system**
- 🚀 **Prepare your environment for building production-grade AI apps**

## 💡 Why This Matters

Modern AI applications aren't just about language models. They require a coordinated ecosystem:
- **LLMs** for intelligence (LLM Gateway + tk-llm)
- **Vector databases** for semantic memory (Qdrant)
- **Observability** for debugging (Langfuse)
- **Experiment tracking** for iteration (MLflow)
- **Relational storage** for structured data (PostgreSQL)
- **Caching** for performance (Valkey)
- **Messaging** for coordination (NATS)

## 🏗️ The AI Application Architecture

```mermaid
graph LR
    A[👤 User Query] --> B[🤖 AI Research Assistant]
    B --> C[🔤 LLM Gateway<br/>Chat + Embeddings]
    B --> D[🗄️ Qdrant<br/>Vectors]
    B --> E[🐘 PostgreSQL<br/>Metadata]
    B --> F[⚡ Valkey<br/>Cache]
    C --> G[📈 Langfuse<br/>Observability]
    B --> H[📡 NATS<br/>Multi-Agent]
    B --> I[🔬 MLflow<br/>Experiments]
    
    style A fill:#e1f5ff,color:#1a1a1a
    style B fill:#fff4e6,color:#1a1a1a
    style C fill:#f3e5f5,color:#1a1a1a
    style D fill:#e8f5e9,color:#1a1a1a
    style E fill:#e3f2fd,color:#1a1a1a
    style F fill:#fff3e0,color:#1a1a1a
    style G fill:#fce4ec,color:#1a1a1a
    style H fill:#e0f2f1,color:#1a1a1a
    style I fill:#f3e5ab,color:#1a1a1a
```

### 🔄 How Data Flows Through the System

**When a user asks**: *"What are the latest papers on parameter-efficient fine-tuning?"*

1. 🔤 **LLM Gateway** converts the question into a vector embedding (via TEI backend)
2. 🗄️ **Qdrant** searches for similar paper chunks using vector similarity
3. 🐘 **PostgreSQL** fetches paper metadata (title, authors, dates)
4. ⚡ **Valkey** checks if we've answered this question before (cache hit!)
5. 🔤 **LLM Gateway** generates a natural language answer using retrieved context (via Ollama/vLLM backend)
6. 📈 **Langfuse** logs the entire interaction (tokens, latency, quality)
7. 📡 **NATS** notifies other agents about the query (for multi-agent systems)
8. 🔬 **MLflow** tracks if this query is part of an A/B test

**All of this happens in milliseconds** ⚡

---

Let's validate each service and understand its specific role!

In [1]:
# Helper functions for colored output
from IPython.display import display, HTML

def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def skip(msg):
    display(HTML(f'<span style="color: #95a5a6;">⊘ {msg}</span>'))

---
## 🔤 Service 1: LLM Gateway - Your Local AI Engine

### 🎯 What is the LLM Gateway?

The Thinkube LLM Gateway gives you a single OpenAI-compatible API for all your **locally deployed** language models. Think of it as having your own private AI service — models run on your GPUs, no API costs, no data leaving your network.

Unlike cloud APIs where you pay per token, Thinkube downloads open-source models (Qwen, Llama, Gemma, etc.) and serves them from your own hardware. The gateway routes your requests to the right backend automatically.

### 🔧 What is `tk-llm`?

`tk-llm` is Thinkube's Python SDK for managing local LLMs. It's pre-installed in every Jupyter environment and code-server. With it, you can:

```python
from tk_llm import LLMClient, get_openai_client

# Management: discover, load, unload models
llm = LLMClient()
llm.list_models()          # What models are available?
llm.gpu_status()           # How much GPU memory is free?
llm.load_model("Qwen/...")  # Load a model onto a GPU
llm.unload_model("Qwen/...") # Free the GPU when done

# Inference: standard OpenAI client — works with LangChain, AG2, etc.
client = get_openai_client()
client.chat.completions.create(...)  # Chat
client.embeddings.create(...)        # Embeddings
```

### 💪 Core Capabilities

- **🔄 Model Management**: Load, unload, and switch between models on demand — the gateway handles routing
- **🖥️ GPU Awareness**: Check memory, utilization, and available slots before loading a model
- **⚡ Multiple Backends**: Ollama (quantized GGUF), vLLM (full precision), TensorRT-LLM (NVIDIA optimized), TEI (embeddings)
- **📥 Embeddings**: Run text embeddings locally via TEI for RAG pipelines — same API, same gateway
- **🔌 OpenAI Compatible**: Your existing code that works with OpenAI works here — LangChain, AG2, CrewAI, any SDK

### 🔬 In the Research Assistant

- **📥 Embeddings**: Convert paper text into vectors for semantic search (notebook 02)
- **💬 Chat**: Answer questions about research papers using RAG (notebook 02)
- **✍️ Summarization**: Generate paper abstracts and key findings
- **🤖 Agents**: Power the debate agents that argue about research approaches (notebook 03)

### 🔍 What We're Testing

✅ **Gateway Connectivity** - Can we reach the LLM Gateway and list models?  
✅ **GPU Status** - Are GPUs available with free slots for inference?  
✅ **Chat Completion** - Can we generate text with a loaded model?  
✅ **Embeddings** - Can we generate vectors for RAG?

> **Note**: Models need to be loaded before inference works. See notebook `01-register-litellm.ipynb` for the complete model management workflow.

In [2]:
import os
from tk_llm import LLMClient, get_openai_client

try:
    llm = LLMClient()
    
    # Check available models
    models = llm.list_models(state="available")
    chat_models = [m for m in models.models if m.task == "text-generation"]
    embed_models = [m for m in models.models if m.task == "feature-extraction"]
    
    success(f"Connected to LLM Gateway")
    info(f"Chat models loaded: {', '.join(m.id for m in chat_models) if chat_models else 'none — load one in notebook 01'}")
    info(f"Embedding models loaded: {', '.join(m.id for m in embed_models) if embed_models else 'none — load one in notebook 01'}")
    
    # Check GPU status
    gpu = llm.gpu_status()
    for node in gpu.nodes:
        info(f"GPU: {node.name} — {node.gpu_product} — {node.available_slots}/{node.total_slots} slots free — {node.real_available_gb:.0f}GB available")
    
    # Test chat if a model is loaded
    if chat_models:
        client = get_openai_client()
        response = client.chat.completions.create(
            model=chat_models[0].id,
            messages=[{"role": "user", "content": "Say 'hello' in one word."}],
            max_tokens=256,
        )
        # A reasoning model spends tokens thinking before it answers; with a
        # small budget the visible answer can be empty although the call
        # succeeded. The budget is generous, and an empty answer is reported
        # as what it is rather than as a gateway failure.
        answer = (response.choices[0].message.content or "").strip()
        if answer:
            success(f"Chat test passed: {answer[:50]}")
        else:
            info("Chat test: the call succeeded but the visible answer was empty — the model spent the budget thinking (notebook 01 explains max_tokens)")
    
    # Test embeddings if an embedding model is loaded
    if embed_models:
        client = get_openai_client()
        emb = client.embeddings.create(
            model=embed_models[0].id,
            input="test embedding"
        )
        success(f"Embedding test passed: {len(emb.data[0].embedding)} dimensions")

except Exception as e:
    error(f"LLM Gateway connection failed: {e}")

---
## 🗄️ Service 2: Qdrant - Your Semantic Memory Engine

### 🎯 What is Qdrant?

Qdrant (pronounced "quadrant") is a **vector database** optimized for similarity search. It stores embeddings (numeric representations of text) and finds the most similar ones blazingly fast.

### 💪 Core Capabilities

- **🔍 Semantic Search**: Find content by meaning, not just keywords
  - Query: "parameter efficient training" → Finds papers about LoRA, QLoRA, AdaLoRA
- **⚡ High Performance**: Millisecond search over millions of vectors using HNSW algorithm
- **🏷️ Metadata Filtering**: Combine semantic + structured search
  - "Papers about transformers published after 2023 by Google authors"
- **📦 Collections**: Organize vectors into separate namespaces (papers, users, etc.)

### 🧪 Real-World Example

```python
# Store a paper chunk
qdrant.upsert(
    collection="research_papers",
    points=[{
        "id": "chunk_123",
        "vector": [0.1, 0.2, ...],  # 768-dim embedding
        "payload": {"paper": "LoRA", "section": "Introduction"}
    }]
)

# Search by similarity
results = qdrant.search(
    collection="research_papers",
    query_vector=[0.15, 0.19, ...],  # User's question embedded
    limit=5
)
```

### 🔬 In the Research Assistant

- **📚 Paper Storage**: Store embeddings for every chunk of every paper
- **🔎 RAG Retrieval**: Find relevant paper sections for user questions
- **📊 Similarity Analysis**: "Find papers similar to this one"
- **🎯 Hybrid Search**: Combine vector similarity with filters (date, author, topic)

### 🔍 What We're Testing

✅ **Connectivity** - Can we reach the Qdrant server?  
✅ **Collections** - Are there existing vector collections?  
✅ **Search** - Can we perform similarity search?

> **Fun Fact**: Qdrant can handle 100M+ vectors while maintaining sub-20ms query latency! ⚡

In [5]:
qdrant_url = os.environ.get('QDRANT_URL')

if not qdrant_url:
    skip("Qdrant not configured - set QDRANT_URL")
else:
    try:
        from qdrant_client import QdrantClient
        from qdrant_client.models import Distance, VectorParams, PointStruct
        import uuid
        
        client = QdrantClient(
            url=qdrant_url,
            port=443,
            https=True,
            verify=False
        )
        
        collections = client.get_collections()
        success(f"Connected to Qdrant at {qdrant_url}")
        info(f"Existing collections: {len(collections.collections)}")
        
        # Demo: Create a test collection, add vectors, search
        test_collection = "thinkube_test"
        
        # Create collection (if not exists)
        if not client.collection_exists(test_collection):
            client.create_collection(
                collection_name=test_collection,
                vectors_config=VectorParams(size=4, distance=Distance.COSINE)
            )
        
        # Insert test vectors (simulating paper embeddings)
        client.upsert(
            collection_name=test_collection,
            points=[
                PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={"title": "LoRA paper"}),
                PointStruct(id=2, vector=[0.2, 0.3, 0.4, 0.5], payload={"title": "QLoRA paper"}),
            ]
        )
        
        # Search for similar vectors (query_points replaces deprecated search in v1.18+)
        results = client.query_points(
            collection_name=test_collection,
            query=[0.15, 0.25, 0.35, 0.45],
            limit=2
        )
        
        info(f"Search test - found {len(results.points)} similar documents")
        for r in results.points:
            info(f"  - {r.payload['title']} (score: {r.score:.3f})")
        
        # Cleanup
        client.delete_collection(test_collection)
        
    except Exception as e:
        error(f"Qdrant connection failed: {e}")

---
## 📈 Service 3: Langfuse - Your LLM Debugging Microscope

### 🎯 What is Langfuse?

Langfuse is an **observability platform** specifically designed for LLM applications. It's like having X-ray vision into your AI system's behavior.

### 💪 Core Capabilities

- **🔍 Trace Every Call**: See the complete execution path of complex AI workflows
  - User query → embedding → vector search → LLM generation → response
- **💰 Cost Tracking**: Know exactly how much each request costs
  - "This RAG query used 1,234 tokens = $0.0185"
- **⏱️ Performance Monitoring**: Identify slow components
  - "LLM took 2.3s, but vector search only 15ms"
- **🎯 Quality Evaluation**: Score response quality and track improvements
- **🐛 Error Analysis**: Debug failed requests with full context

### 🧪 Real-World Example

```python
# Langfuse automatically traces LangChain & CrewAI!
from langfuse.callback import CallbackHandler
langfuse_handler = CallbackHandler()

# Every LLM call is now logged
chain.invoke({"question": "What is LoRA?"}, callbacks=[langfuse_handler])

# View in Langfuse UI:
# ├─ embed_query (nomic-embed, 25 tokens, 50ms, $0.0001)
# ├─ vector_search (Qdrant, 5 results, 12ms)
# └─ generate_answer (gpt-oss-20b, 150 tokens, 1.2s, $0.003)
```

### 🔬 In the Research Assistant

- **🔎 RAG Pipeline Visibility**: See embed → search → generate steps
- **🤖 Multi-Agent Debugging**: Track which agent did what and why
- **📊 A/B Testing**: Compare different prompts or models
- **💸 Cost Control**: Monitor spending per user, per query type
- **🎓 Quality Improvement**: Find and fix poor responses

### 🔍 What We're Testing

✅ **Connectivity** - Can we authenticate with Langfuse?  
✅ **API Access** - Can we create traces and spans?  
✅ **Observability Ready** - Is it ready to track our RAG pipeline?

> **Pro Tip**: Always enable Langfuse tracing during development. It saves hours of debugging! 🕵️

In [6]:
langfuse_host = os.environ.get('LANGFUSE_HOST')
langfuse_public_key = os.environ.get('LANGFUSE_PUBLIC_KEY')
langfuse_secret_key = os.environ.get('LANGFUSE_SECRET_KEY')

if not all([langfuse_host, langfuse_public_key, langfuse_secret_key]):
    skip("Langfuse not configured - set LANGFUSE_HOST, LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY")
else:
    try:
        from langfuse import Langfuse
        
        langfuse = Langfuse(
            public_key=langfuse_public_key,
            secret_key=langfuse_secret_key,
            host=langfuse_host
        )
        
        # Verify authentication
        langfuse.auth_check()
        success(f"Connected to Langfuse at {langfuse_host}")
        
        # Note: Langfuse is ready to track LLM calls in RAG pipeline
        info("LLM observability ready for notebooks 02-04")
        info("  • Tracks RAG pipeline steps (embed → search → generate)")
        info("  • Records token usage and latency")
        info("  • Links traces to experiments")
        
    except Exception as e:
        error(f"Langfuse connection failed: {e}")

---
## 🔬 Service 4: MLflow - Your ML Experiment Laboratory

### 🎯 What is MLflow?

MLflow is an **experiment tracking platform** that helps you manage the entire machine learning lifecycle - from experimentation to production deployment.

### 💪 Core Capabilities

- **📊 Experiment Tracking**: Log parameters, metrics, and artifacts
  - Learning rate: 2e-4, Batch size: 16, Loss: 0.23, Accuracy: 94%
- **🗂️ Model Registry**: Version and organize trained models
  - "LoRA-v1" → "LoRA-v2-improved" → "LoRA-v3-production"
- **📈 Metric Visualization**: Compare experiments with charts and tables
- **🔗 Artifact Storage**: Save model checkpoints, datasets, plots
- **🚀 Model Deployment**: Serve models with MLflow's deployment APIs

### 🧪 Real-World Example

```python
import mlflow

# Start an experiment
with mlflow.start_run(run_name="qlora-finetuning"):
    # Log hyperparameters
    mlflow.log_params({
        "learning_rate": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32
    })
    
    # Train model...
    
    # Log metrics during training
    mlflow.log_metrics({
        "train_loss": 0.23,
        "eval_accuracy": 0.94
    })
    
    # Save the model
    mlflow.log_artifact("model.safetensors")
```

### 🔬 In the Research Assistant

- **🎯 Fine-Tuning Experiments**: Track all hyperparameters and results
  - Compare: LoRA rank 8 vs 16 vs 32 on research papers
- **📊 Model Comparison**: Which base model works best? (Llama vs Qwen vs Phi)
- **🔗 Research Linkage**: Link papers to experiments
  - "Paper arxiv:2305.14314 inspired experiment #42"
- **📈 Performance Tracking**: Monitor model quality over time
- **🚀 Model Deployment**: Register and deploy fine-tuned models to the LLM Gateway

### 🔍 What We're Testing

✅ **Connectivity** - Can we reach the MLflow tracking server?  
✅ **Experiments** - Can we create experiments and runs?  
✅ **Logging** - Can we log parameters and metrics?

> **Best Practice**: Always log your experiments! Future-you will thank you when comparing 50 different runs. 📝

In [7]:
mlflow_uri = os.environ.get('MLFLOW_TRACKING_URI')

if not mlflow_uri:
    skip("MLflow not configured - set MLFLOW_TRACKING_URI")
else:
    try:
        import mlflow
        import requests
        import urllib3
        urllib3.disable_warnings()
        
        # MLflow with Keycloak OAuth
        mlflow_username = os.environ.get('MLFLOW_AUTH_USERNAME')
        mlflow_password = os.environ.get('MLFLOW_AUTH_PASSWORD')
        token_url = os.environ.get('MLFLOW_KEYCLOAK_TOKEN_URL')
        client_id = os.environ.get('MLFLOW_KEYCLOAK_CLIENT_ID')
        client_secret = os.environ.get('MLFLOW_CLIENT_SECRET')
        
        if not all([mlflow_username, mlflow_password, token_url, client_id, client_secret]):
            skip("MLflow OAuth not fully configured")
        else:
            # Get OAuth token
            token_response = requests.post(
                token_url,
                data={
                    'grant_type': 'password',
                    'client_id': client_id,
                    'client_secret': client_secret,
                    'username': mlflow_username,
                    'password': mlflow_password
                },
                verify=False
            )
            
            if token_response.status_code == 200:
                mlflow_token = token_response.json().get('access_token')
                os.environ['MLFLOW_TRACKING_TOKEN'] = mlflow_token
                mlflow.set_tracking_uri(mlflow_uri)
                
                experiments = mlflow.search_experiments(max_results=5)
                success(f"Connected to MLflow at {mlflow_uri}")
                info(f"Experiments: {len(experiments)}")
                
                # Demo: Log a test run (like we would for fine-tuning)
                mlflow.set_experiment("research-assistant-validation")
                with mlflow.start_run(run_name="platform-test"):
                    mlflow.log_param("model", "test")
                    mlflow.log_metric("accuracy", 0.95)
                info("Created test experiment run")
            else:
                error(f"MLflow OAuth failed: {token_response.status_code}")
                
    except Exception as e:
        error(f"MLflow connection failed: {e}")

2026/05/12 23:29:06 INFO mlflow.tracking.fluent: Experiment with name 'research-assistant-validation' does not exist. Creating a new experiment.


🏃 View run platform-test at: https://mlflow.thinkube.com/#/experiments/2/runs/3f39754cb45f42008f69d5e8f1f91a73
🧪 View experiment at: https://mlflow.thinkube.com/#/experiments/2


---
## 🐘 Service 5: PostgreSQL - Your Structured Data Foundation

### 🎯 What is PostgreSQL?

PostgreSQL is a **powerful relational database** that excels at storing structured data with complex relationships. It's been battle-tested for over 30 years!

### 💪 Core Capabilities

- **📊 ACID Transactions**: Guaranteed data consistency and reliability
- **🔗 Relational Queries**: Join data across tables with SQL
  - "Find all papers by authors who published in NeurIPS 2024"
- **🔍 Full-Text Search**: Search paper titles and abstracts efficiently
- **🗂️ JSON Support**: Store flexible metadata alongside structured data
- **📈 Analytics**: Aggregate and analyze large datasets

### 🧪 Real-World Example

```sql
-- Store paper metadata
CREATE TABLE papers (
    id SERIAL PRIMARY KEY,
    arxiv_id VARCHAR(20) UNIQUE,
    title TEXT,
    authors JSONB,  -- Flexible author list
    abstract TEXT,
    published_date DATE,
    processed BOOLEAN DEFAULT FALSE
);

-- Complex query: Papers from 2024 with "LoRA" in title
SELECT title, authors->>0 AS first_author, published_date
FROM papers
WHERE published_date >= '2024-01-01'
  AND title ILIKE '%LoRA%'
ORDER BY published_date DESC;
```

### 🔬 In the Research Assistant

- **📚 Paper Catalog**: Store titles, authors, abstracts, ArXiv IDs
- **✅ Processing Status**: Track which papers have been embedded
- **🔗 Relationships**: Link papers → chunks → embeddings → experiments
- **📊 Analytics**: "How many papers per month?", "Top authors?"
- **👤 User Management**: Store user preferences and query history

### 🔍 What We're Testing

✅ **Connectivity** - Can we connect to PostgreSQL?  
✅ **Database Access** - Can we query the database?  
✅ **Version Check** - Is it a recent version with modern features?

> **Why Not Just Use Qdrant?** Qdrant is for vectors, PostgreSQL is for structured relationships. Use both! 🤝

In [8]:
postgres_host = os.environ.get('POSTGRES_HOST')
postgres_password = os.environ.get('POSTGRES_PASSWORD')

if not postgres_host or not postgres_password:
    skip("PostgreSQL not configured - set POSTGRES_HOST and POSTGRES_PASSWORD")
else:
    try:
        import psycopg2
        
        # Connect to default postgres database to check connectivity
        conn = psycopg2.connect(
            host=postgres_host,
            port=int(os.environ.get('POSTGRES_PORT', 5432)),
            database='postgres',  # Connect to default database
            user=os.environ.get('POSTGRES_USER', 'postgres'),
            password=postgres_password
        )
        
        cursor = conn.cursor()
        cursor.execute('SELECT version();')
        version = cursor.fetchone()[0]
        
        success(f"Connected to PostgreSQL at {postgres_host}")
        info(f"Version: {version.split(',')[0]}")
        
        # Demo: Show how we'd store paper metadata
        info("Example schema for Research Assistant:")
        info("  papers(id, arxiv_id, title, authors, abstract, published_date)")
        info("  chunks(id, paper_id, content, embedding_id, chunk_index)")
        info("  experiments(id, paper_id, mlflow_run_id, notes)")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        error(f"PostgreSQL connection failed: {e}")

---
## ⚡ Service 6: Valkey - Your High-Speed Cache Layer

### 🎯 What is Valkey?

Valkey is a **high-performance in-memory data store** (Redis-compatible fork). Think of it as super-fast RAM-based storage for frequently accessed data.

### 💪 Core Capabilities

- **🚀 Blazing Fast**: Sub-millisecond latency (100x faster than disk-based databases)
- **💾 Key-Value Storage**: Simple but powerful data structures
  - Strings, hashes, lists, sets, sorted sets
- **⏱️ TTL Support**: Automatically expire cached data
  - Cache embeddings for 1 hour, then refresh
- **🔄 Pub/Sub**: Real-time messaging (lightweight alternative to NATS)
- **📊 Counters**: Track API usage, rate limits

### 🧪 Real-World Example

```python
import redis

# Cache expensive embeddings
valkey.set(
    "embedding:what_is_lora",
    json.dumps(embedding_vector),
    ex=3600  # Expire in 1 hour
)

# Retrieve from cache (avoid re-computing)
cached = valkey.get("embedding:what_is_lora")
if cached:
    embedding = json.loads(cached)  # 0.5ms
else:
    embedding = llm.embed("What is LoRA?")  # 50ms

# Rate limiting
requests = valkey.incr(f"user:{user_id}:requests")
if requests > 100:
    raise TooManyRequestsError()
```

### 🔬 In the Research Assistant

- **📥 Embedding Cache**: Avoid re-computing embeddings for common queries
  - "What is LoRA?" asked 100 times? Compute once, cache forever
- **💬 Session State**: Store conversation history for multi-turn chat
- **📊 Rate Limiting**: Prevent abuse (max 100 queries/hour per user)
- **🎯 Response Cache**: Cache complete answers to popular questions
- **⚡ Hot Data**: Store frequently accessed paper metadata

### 🔍 What We're Testing

✅ **Connectivity** - Can we connect to Valkey?  
✅ **Read/Write** - Can we store and retrieve data?  
✅ **Performance** - Is it responding in milliseconds?

> **Performance Tip**: Cache embeddings aggressively! They're expensive to compute but cheap to store. 💰

In [10]:
valkey_host = os.environ.get('VALKEY_HOST')
valkey_port = os.environ.get('VALKEY_PORT')
valkey_password = os.environ.get('VALKEY_PASSWORD', '')

if not valkey_host or not valkey_port:
    skip("Valkey not configured - set VALKEY_HOST and VALKEY_PORT")
else:
    try:
        import redis
        import json
        
        client = redis.Redis(
            host=valkey_host,
            port=int(valkey_port),
            password=valkey_password,
            decode_responses=True
        )
        
        client.ping()
        server_info = client.info('server')
        
        success(f"Connected to Valkey at {valkey_host}")
        info(f"Version: {server_info.get('redis_version', 'unknown')}")
        
        # Demo: Cache a paper summary (like we would in the Research Assistant)
        paper_id = "arxiv:2106.09685"  # LoRA paper
        cache_key = f"summary:{paper_id}"
        
        # Simulate caching a summary
        summary = {"title": "LoRA", "summary": "Low-rank adaptation for efficient fine-tuning"}
        client.setex(cache_key, 3600, json.dumps(summary))  # Cache for 1 hour
        
        # Retrieve from cache
        cached = json.loads(client.get(cache_key))
        info(f"Cache test - stored and retrieved: {cached['title']}")
        
        # Cleanup
        client.delete(cache_key)
        
    except Exception as e:
        error(f"Valkey connection failed: {e}")

---
## 📡 Service 7: NATS - Your Multi-Agent Message Highway

### 🎯 What is NATS?

NATS is a **lightweight, high-performance message broker** designed for cloud-native systems. It enables microservices and agents to communicate asynchronously.

### 💪 Core Capabilities

- **📨 Pub/Sub**: Broadcast messages to multiple subscribers
  - "New paper published!" → All agents notified instantly
- **🎯 Request/Reply**: Direct agent-to-agent communication
  - Researcher agent: "Summarize this paper" → Writer agent: "Done!"
- **⚡ High Throughput**: Handle millions of messages per second
- **🌐 Distributed Systems**: Works across multiple servers/regions
- **📦 Message Queues**: Ensure no message is lost

### 🧪 Real-World Example

```python
import nats

# Agent 1: Paper Ingestion Agent
nc = await nats.connect("nats://localhost:4222")
await nc.publish("papers.new", json.dumps({
    "arxiv_id": "2305.14314",
    "title": "QLoRA: Efficient Finetuning"
}))

# Agent 2: Summarization Agent (listening)
async def handle_new_paper(msg):
    paper = json.loads(msg.data)
    summary = await summarize(paper)
    await nc.publish("papers.summarized", summary)

await nc.subscribe("papers.new", cb=handle_new_paper)

# Agent 3: Notification Agent (listening to summaries)
await nc.subscribe("papers.summarized", cb=notify_users)
```

### 🔬 In the Research Assistant

- **🤖 Multi-Agent Coordination**: Agents communicate without tight coupling
  - **Researcher Agent**: Finds relevant papers → publishes to `research.found`
  - **Analyst Agent**: Listens to `research.found` → analyzes → publishes to `research.analyzed`
  - **Writer Agent**: Listens to `research.analyzed` → writes report → publishes to `research.completed`
- **📊 Event-Driven Architecture**: Loose coupling between components
- **⚡ Async Processing**: Agents work in parallel, not sequentially
- **🔄 Scalability**: Add more agents without changing existing code

### 🔍 What We're Testing

✅ **Connectivity** - Can we connect to NATS?  
✅ **Pub/Sub** - Can we publish and receive messages?  
✅ **Latency** - Is message delivery fast?

> **Architecture Insight**: NATS enables you to build systems where agents don't need to know about each other - just the message topics! 🎯

In [11]:
nats_url = os.environ.get('NATS_URL')

if not nats_url:
    skip("NATS not configured - set NATS_URL")
else:
    try:
        import nats
        import asyncio
        import json
        
        async def test_nats():
            nc = await nats.connect(nats_url)
            
            # Demo: Simulate multi-agent messaging
            received_messages = []
            
            async def message_handler(msg):
                received_messages.append(json.loads(msg.data.decode()))
            
            # Experiment Tracker subscribes to new papers
            sub = await nc.subscribe("papers.new", cb=message_handler)
            
            # Paper Summarizer publishes a new paper
            paper_event = {"arxiv_id": "2106.09685", "title": "LoRA", "action": "summarized"}
            await nc.publish("papers.new", json.dumps(paper_event).encode())
            
            # Wait for message delivery
            await asyncio.sleep(0.1)
            await sub.unsubscribe()
            await nc.close()
            
            return received_messages
        
        messages = await test_nats()
        
        success(f"Connected to NATS at {nats_url}")
        if messages:
            info(f"Pub/sub test - received: {messages[0]['title']}")
        info("Multi-agent messaging ready")
        
    except Exception as e:
        error(f"NATS connection failed: {e}")

---
## 🎓 Summary: What the Next Notebooks Build on These Services

You have checked seven services. The research assistant uses four of them; PostgreSQL, Valkey and NATS are checked because your own notebooks and apps can use them too.

### 📚 The path from here

```mermaid
graph TD
    A[🔤 Notebook 01<br/>Load a chat model and an embedding model] --> B[📊 Notebook 02<br/>Index arXiv papers and answer with sources]
    B --> C[⚖️ Notebook 03<br/>Two agents debate from the index]
    C --> D[🦓 zebra-grpo<br/>Fine-tune, register and serve your own model]

    B --> Q[🗄️ Qdrant<br/>rl_reasoning_papers]
    A --> G[🔤 LLM Gateway<br/>chat and embeddings]
    B --> G
    C --> G
    B --> L[📈 Langfuse<br/>traces]
    C --> L
    D --> M[🔬 Thinkube Experiments<br/>run and registered model]
    M --> G

    style A fill:#f3e5f5,color:#1a1a1a
    style B fill:#fff4e6,color:#1a1a1a
    style C fill:#f1f8e9,color:#1a1a1a
    style D fill:#fef5e7,color:#1a1a1a
    style Q fill:#e8f5e9,color:#1a1a1a
    style G fill:#e1f5ff,color:#1a1a1a
    style L fill:#fce4ec,color:#1a1a1a
    style M fill:#f3e5ab,color:#1a1a1a
```

### 🔄 What each notebook does

#### **Notebook 01 - Working with Local LLMs**
Checks the GPU slots, loads a chat model and an embedding model through the **LLM Gateway** with `tk-llm`, and tests both. Leave them loaded for the next notebooks.

#### **Notebook 02 - RAG Pipeline**
Fetches the 2025–2026 papers on reasoning post-training from arXiv, splits them into chunks, embeds them through the **LLM Gateway**, stores the vectors in **Qdrant** (collection `rl_reasoning_papers`) and answers questions with sources. **Langfuse** records the traces.

#### **Notebook 03 - AI Debate Arena (AG2)**
Two agents argue whether to post-train a small model with GRPO or by distillation, each citing papers from the **Qdrant** index; a judge weighs the arguments and declares a winner. Every model call goes through the **LLM Gateway** and is traced in **Langfuse**.

#### **zebra-grpo - Fine-Tune and Deploy Your Own Model**
Runs the first of those techniques end to end with the `fine-tuning` kernel on one GPU: trains a LoRA adapter with GRPO, records the run in **Thinkube Experiments** (MLflow), registers the merged model and serves it through the **LLM Gateway**.

---

### 🎯 What You've Accomplished

✅ **LLM Gateway** - Chat and embedding models are loaded and ready  
✅ **Qdrant** - Vector database is operational  
✅ **Langfuse** - Observability platform is tracking-ready  
✅ **MLflow** - Experiment tracking is configured  
✅ **PostgreSQL** - Relational database is connected  
✅ **Valkey** - Caching layer is functional  
✅ **NATS** - Message broker is ready  

---

### 🚀 Next Steps

1. **Next Notebook**: `01-register-litellm.ipynb`  
   - Discover available models and check GPU resources
   - Load and unload models via `tk-llm` SDK
   - Test chat completions and embeddings

2. **Then Continue to**: `02-langchain-rag.ipynb`  
   - Build the RAG pipeline
   - Index research papers from arXiv
   - Query your knowledge base

💡 **Remember**: All models run locally on your GPUs — no external API calls, no data leaving your network!